# Fase 08 — Extracción estructurada de hallazgos

Esta fase convierte los fragmentos de las fuentes incluidas en hallazgos independientes, verificables y trazables. La unidad de análisis ya no es el documento completo ni el fragmento, sino el **hallazgo**.

Principios:

- cada hallazgo conserva `source_id`, `chunk_id`, páginas y extracto de respaldo;
- una recomendación no se codifica como resultado observado;
- una proyección o escenario no se codifica como tendencia observada;
- una actividad o producto de proyecto no se codifica como resultado o impacto sin evaluación;
- una propuesta, autorización o compromiso no prueba implementación ni efectividad;
- los fragmentos superpuestos pueden producir duplicados, que deben marcarse antes de la síntesis;
- las extracciones asistidas permanecen como `not_reviewed` hasta validación humana.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from evidence_review.evidence_extraction import (
    annotate_near_duplicate_findings,
    build_source_finding_summary,
    enrich_extraction_corpus,
    export_structured_finding_prompts_jsonl,
    extraction_summary,
    initialise_extraction_queue,
    initialise_findings_working,
    load_finding_extraction_config,
    merge_extraction_queue,
    merge_findings_working,
    read_csv_robust,
    read_structured_finding_responses_jsonl,
    split_structured_finding_outputs,
    synchronise_queue_with_findings,
    validate_structured_finding_outputs,
)

CONFIG_PATH = ROOT / "config" / "finding_extraction.yml"
config = load_finding_extraction_config(CONFIG_PATH, project_root=ROOT)
paths = config["paths"]

INTERIM = ROOT / "data" / "interim"
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Configuration: {CONFIG_PATH.relative_to(ROOT)}")


## 1. Cargar el corpus de extracción y los metadatos

La entrada principal contiene los 350 fragmentos de las 16 fuentes incluidas en la fase 07. Los metadatos de fuente se incorporan por `source_id`.


In [ ]:
corpus_path = ROOT / paths["corpus_csv"]
metadata_path = ROOT / paths["source_metadata_csv"]
queue_path = ROOT / paths["queue_working_csv"]
findings_path = ROOT / paths["findings_working_csv"]

raw_corpus, corpus_encoding = read_csv_robust(corpus_path)
metadata = pd.DataFrame()
metadata_encoding = "not_loaded"
if metadata_path.exists():
    metadata, metadata_encoding = read_csv_robust(metadata_path)

corpus = enrich_extraction_corpus(raw_corpus, metadata)

existing_queue = pd.DataFrame()
if queue_path.exists():
    existing_queue, _ = read_csv_robust(queue_path)

existing_findings = pd.DataFrame()
if findings_path.exists():
    existing_findings, _ = read_csv_robust(findings_path)

print(f"Extraction chunks: {len(corpus)} [{corpus_encoding}]")
print(f"Included sources: {corpus['source_id'].nunique()}")
print(f"Source metadata: {len(metadata)} [{metadata_encoding}]")
print(f"Existing queue rows: {len(existing_queue)}")
print(f"Existing finding rows: {len(existing_findings)}")


## 2. Inicializar la cola y la hoja de hallazgos

La cola conserva una decisión por fragmento, incluso cuando no existe un hallazgo codificable. Las ediciones humanas previas se preservan por `chunk_id` y `finding_id`.


In [ ]:
queue = initialise_extraction_queue(corpus, existing_queue)
findings = initialise_findings_working(existing_findings)

queue.to_csv(queue_path, index=False, encoding="utf-8-sig")
findings.to_csv(findings_path, index=False, encoding="utf-8-sig")

print(f"Queue rows: {len(queue)}")
print(f"Finding rows: {len(findings)}")
display(extraction_summary(queue, findings))
display(queue.head(10))


## 3. Generar prompts auditables

Se genera un prompt JSONL por fragmento. Esta celda **no llama a ningún modelo externo**. Cada prompt contiene el vocabulario controlado, las distinciones metodológicas y el texto con páginas.


In [ ]:
prompts_path = ROOT / paths["prompts_jsonl"]
export_structured_finding_prompts_jsonl(corpus, config, prompts_path)

print(f"Prompts generated: {len(corpus)}")
print(f"Saved: {prompts_path.relative_to(ROOT)}")


## 4. Importar respuestas asistidas opcionales

El archivo esperado es `structured_finding_model_responses.jsonl`, con un objeto por fragmento. La importación está desactivada por defecto. Las filas humanas `accepted`, `corrected` o `rejected` no se sobrescriben salvo activación explícita.


In [ ]:
IMPORT_MODEL_RESPONSES = False
OVERWRITE_HUMAN_VALIDATED = False

responses_path = ROOT / paths["model_responses_jsonl"]

if IMPORT_MODEL_RESPONSES:
    if not responses_path.exists():
        raise FileNotFoundError(responses_path)

    queue_updates, incoming_findings = read_structured_finding_responses_jsonl(
        responses_path,
        corpus,
        config,
    )

    queue = merge_extraction_queue(
        queue,
        queue_updates,
        overwrite_human_validated=OVERWRITE_HUMAN_VALIDATED,
    )
    findings = merge_findings_working(
        findings,
        incoming_findings,
        overwrite_human_validated=OVERWRITE_HUMAN_VALIDATED,
    )

    # Persist imported responses immediately so later cells can safely reload
    # the latest reviewed state from disk.
    queue.to_csv(queue_path, index=False, encoding="utf-8-sig")
    findings.to_csv(findings_path, index=False, encoding="utf-8-sig")

    print(f"Imported chunk decisions: {len(queue_updates)}")
    print(f"Imported candidate findings: {len(incoming_findings)}")
    print(f"Saved queue: {queue_path}")
    print(f"Saved findings: {findings_path}")
else:
    print(
        "Model-response import disabled. The reviewed working CSV on disk "
        "will be reloaded before validation and export."
    )


## 5. Marcar duplicados y sincronizar la cola

Los fragmentos tienen solapamiento. La detección compara hallazgos de la misma fuente con páginas cercanas y vocabulario altamente coincidente. Las coincidencias se marcan como `possible_duplicate`; no se eliminan automáticamente.


In [ ]:
# Always reload the working CSVs from disk before validation/export.
# This prevents stale notebook variables from overwriting human edits made in Excel.
disk_queue, queue_encoding = read_csv_robust(queue_path)
disk_findings, findings_encoding = read_csv_robust(findings_path)

queue = initialise_extraction_queue(corpus, disk_queue)
findings = initialise_findings_working(disk_findings)

print(f"Reloaded queue from: {queue_path} [{queue_encoding}]")
print(f"Reloaded findings from: {findings_path} [{findings_encoding}]")
print(
    findings["human_validation_status"]
    .fillna("")
    .astype(str)
    .str.strip()
    .replace("", "blank")
    .value_counts(dropna=False)
)

findings = annotate_near_duplicate_findings(findings, config)
queue = synchronise_queue_with_findings(queue, findings)

queue.to_csv(queue_path, index=False, encoding="utf-8-sig")
findings.to_csv(findings_path, index=False, encoding="utf-8-sig")

display(extraction_summary(queue, findings))
display(
    findings.loc[
        findings["deduplication_status"].isin(["primary", "possible_duplicate"]),
        [
            "finding_id", "source_id", "chunk_id", "unit_locator",
            "finding_type", "duplicate_group_id", "deduplication_status",
            "evidence_summary",
        ],
    ].head(30)
)


## 6. Validar trazabilidad, lógica y vocabularios

Las filas `pending` son trabajo pendiente y no generan errores. Las validaciones comprueban el vínculo fuente–fragmento, páginas, coincidencia del extracto con el texto, vocabularios, lógica de proyección/implementación y conteos de hallazgos.


In [ ]:
issues = validate_structured_finding_outputs(
    queue,
    findings,
    corpus,
    config,
)

issues_path = ROOT / paths["issues_csv"]
issues.to_csv(issues_path, index=False, encoding="utf-8-sig")

print(f"Validation issues: {len(issues)}")
display(issues.head(100))
display(extraction_summary(queue, findings))


## 7. Exportar hallazgos y auditorías

`structured_findings.csv` excluye hallazgos rechazados y duplicados confirmados. `structured_findings_validated.csv` contiene únicamente filas `accepted` o `corrected`.


In [ ]:
# Reload the persisted reviewed state one final time before creating exports.
queue_disk, _ = read_csv_robust(queue_path)
findings_disk, _ = read_csv_robust(findings_path)
queue = initialise_extraction_queue(corpus, queue_disk)
findings = initialise_findings_working(findings_disk)

groups = split_structured_finding_outputs(queue, findings)
source_summary = build_source_finding_summary(groups["findings"])
flow = extraction_summary(queue, findings)

output_frames = {
    paths["findings_csv"]: groups["findings"],
    paths["validated_findings_csv"]: groups["validated"],
    paths["pending_validation_csv"]: groups["pending_validation"],
    paths["rejected_findings_csv"]: groups["rejected"],
    paths["no_finding_chunks_csv"]: groups["no_finding_chunks"],
    paths["needs_context_chunks_csv"]: groups["needs_context_chunks"],
    paths["extraction_errors_csv"]: groups["extraction_errors"],
    paths["source_summary_csv"]: source_summary,
    paths["flow_summary_csv"]: flow,
}

for relative_path, frame in output_frames.items():
    path = ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{path.relative_to(ROOT)}: {len(frame)}")

display(flow)
display(source_summary.head(20))


## Criterio para avanzar a la fase 09

La evaluación crítica de calidad puede comenzar cuando:

1. `Validation issues = 0`;
2. cada uno de los 350 fragmentos tiene un estado distinto de `pending`;
3. todo fragmento `findings_extracted` tiene al menos un hallazgo;
4. todos los extractos y páginas son verificables;
5. los posibles duplicados han sido revisados;
6. los hallazgos usados en la síntesis están `accepted` o `corrected`;
7. proyecciones, recomendaciones, implementación y efectividad permanecen diferenciadas.

La siguiente fase será:

```text
notebooks/09_appraise_evidence_quality.ipynb
```
